In [1]:
#1. — Setup: Municipality Context
# Purpose:
# Set up the notebook and define the KZN municipality-context input
# and processed-output locations.

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    r"C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA"
)

MUNICIPALITY_INPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "KZN population demographics Cleaned"
)

ELECTION_CONTEXT_DIR = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "KZN_Voter_Prediction_2021-2022 Cleaned"
)

MUNICIPALITY_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "03_municipality_context"
)

MUNICIPALITY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MUNICIPALITY_FILES = {
    "population_demographics": (
        MUNICIPALITY_INPUT_DIR
        / "kzn_municipality_population_demographics_clean.csv"
    ),
    "election_2016": (
        ELECTION_CONTEXT_DIR
        / "IEC_2016_Municipal_Results_clean.csv"
    ),
    "election_2021": (
        ELECTION_CONTEXT_DIR
        / "IEC_2021_Municipal_Results_clean.xls"
    ),
}

MUNICIPALITY_OUTPUT = (
    MUNICIPALITY_OUTPUT_DIR
    / "municipality_context.csv"
)

print("Setup completed successfully.")
print(f"Project root: {PROJECT_ROOT}")
print(f"Municipality input directory: {MUNICIPALITY_INPUT_DIR}")
print(f"Election context directory: {ELECTION_CONTEXT_DIR}")
print(f"Output directory: {MUNICIPALITY_OUTPUT_DIR}")
print()

for name, file_path in MUNICIPALITY_FILES.items():
    print(f"{name}: {file_path}")

print()
print(f"Output file: {MUNICIPALITY_OUTPUT}")

assert PROJECT_ROOT.exists(), "Project root does not exist."
assert MUNICIPALITY_INPUT_DIR.exists(), "Municipality input directory does not exist."
assert ELECTION_CONTEXT_DIR.exists(), "Election context directory does not exist."

for name, file_path in MUNICIPALITY_FILES.items():
    assert file_path.exists(), f"{name} file not found."

print()
print("Path verification: PASSED")

Setup completed successfully.
Project root: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA
Municipality input directory: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\interim\KZN population demographics Cleaned
Election context directory: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\interim\KZN_Voter_Prediction_2021-2022 Cleaned
Output directory: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\03_municipality_context

population_demographics: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\interim\KZN population demographics Cleaned\kzn_municipality_population_demographics_clean.csv
election_2016: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\interim\KZN_Voter_Prediction_2021-2022 Cleaned\IEC_2016_Municipal_Results_clean.csv
election_2021: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\interim\KZN_Voter_Prediction_2021-2022 Cleaned\IEC_2021_Municipal_Results_clean.xls

Output file: C:\Users\Student\Downloads\B

In [2]:
#2. — Load and Inspect Municipality Data
# Purpose:
# Load the municipality demographics and election context datasets
# so we can confirm their structure before deciding what to merge.

municipality_data = {}

for name, file_path in MUNICIPALITY_FILES.items():

    if file_path.suffix.lower() == ".xls":
        df = pd.read_excel(file_path)
    else:
        df = pd.read_csv(file_path)

    municipality_data[name] = df

    print(f"{name} dataset loaded successfully.")
    print(f"Rows: {len(df):,}")
    print(f"Columns: {len(df.columns)}")
    print(f"Columns: {df.columns.tolist()}")
    print()

assert set(municipality_data.keys()) == set(MUNICIPALITY_FILES.keys())

print("All municipality datasets loaded successfully.")
print("Datasets available:", sorted(municipality_data.keys()))

population_demographics dataset loaded successfully.
Rows: 44
Columns: 10
Columns: ['municipality', 'household', 'homeless', 'transient', 'institution', 'urban_area', 'tribal_or_traditional_area', 'farm_area', 'male', 'population']

election_2016 dataset loaded successfully.
Rows: 40,706
Columns: 11
Columns: ['Province', 'Municipality', 'Ward', 'VotingDistrict', 'VotingStationName', 'RegisteredVoters', 'BallotType', 'SpoiltVotes', 'PartyName', 'TotalValidVotes', 'DateGenerated']

WARNING *** file size (24512) not 512 + multiple of sector size (512)
election_2021 dataset loaded successfully.
Rows: 37
Columns: 18
Columns: ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17']

All municipality datasets loaded successfully.
Datasets available: ['election_2016', 'election_2021', 'populat

In [3]:
#3. — Inspect Municipality Dataset Contents
# Purpose:
# Inspect the municipality records and the 2021 election file structure
# before deciding which datasets can safely contribute to the panel.

population_demographics_df = municipality_data["population_demographics"]
election_2016_df = municipality_data["election_2016"]
election_2021_df = municipality_data["election_2021"]

print("POPULATION DEMOGRAPHICS")
print("Municipalities:", population_demographics_df["municipality"].nunique())
print("Population total:", population_demographics_df["population"].sum())
print()
display(population_demographics_df.head())

print()
print("2016 ELECTION DATA")
print("Provinces:", election_2016_df["Province"].unique())
print("Municipalities:", election_2016_df["Municipality"].nunique())
print("Ballot types:", election_2016_df["BallotType"].unique())
print()
display(election_2016_df.head())

print()
print("2021 ELECTION FILE — RAW STRUCTURE")
print(f"Rows: {len(election_2021_df):,}")
print(f"Columns: {len(election_2021_df.columns):,}")
print()

display(election_2021_df.head(10))

print()
print("2021 non-null values by column:")
print(election_2021_df.notna().sum())

assert population_demographics_df["municipality"].nunique() == 44
assert election_2016_df["Province"].eq("KwaZulu-Natal").all()

print()
print("Initial municipality data inspection: PASSED")

POPULATION DEMOGRAPHICS
Municipalities: 44
Population total: 12423906.662785623



,municipality,household,homeless,transient,institution,urban_area,tribal_or_traditional_area,farm_area,male,population
0,Umdoni Local Municipality,99,1,262,39850.271426,104607.218618,11985.390218,74391.521503,82051.358758,156442.880261
1,Umzumbe Local Municipality,56,4,125,0.000000,139029.363876,15.442094,65512.792209,73532.013761,139044.805970
2,UMuziwabantu Local Municipality,54,3,183,12871.211719,100534.222489,2374.091303,54516.210659,61263.314852,115779.525511
3,Ray Nkonyeni Local Municipality,204,1,998,97837.430403,254636.931174,9660.081033,171980.174616,190154.267993,362134.442609
4,uMshwathi Local Municipality,10,1,605,15332.360949,72093.563278,31051.908503,55836.571936,62641.260794,118477.832730



2016 ELECTION DATA
Provinces: ['KwaZulu-Natal']
Municipalities: 1
Ballot types: ['PR' 'Ward']



,Province,Municipality,Ward,VotingDistrict,VotingStationName,RegisteredVoters,BallotType,SpoiltVotes,PartyName,TotalValidVotes,DateGenerated
0,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,1607,PR,72,ACADEMIC CONGRESS UNION,0,8/11/2016 3:58:48 PM
1,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,1607,PR,72,AFRICAN CHRISTIAN DEMOCRATIC PARTY,0,8/11/2016 3:58:48 PM
2,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,1607,PR,72,AFRICAN INDEPENDENT CONGRESS,19,8/11/2016 3:58:48 PM
3,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,1607,PR,72,AFRICAN MANTUNGWA COMMUNITY,0,8/11/2016 3:58:48 PM
4,KwaZulu-Natal,ETH - eThekwini,Ward 59500001,43400179,NOMFIHLELA PRIMARY SCHOOL,1607,PR,72,AFRICAN NATIONAL CONGRESS,1157,8/11/2016 3:58:48 PM



2021 ELECTION FILE — RAW STRUCTURE
Rows: 37
Columns: 18



,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,Detailed DC 40% Ballot Results Report,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,Printed on: 2021/11/22 16:19:57,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Electoral Event:,NaN,NaN,LOCAL GOVERNMENT ELECTION 2021,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Province:,NaN,NaN,KwaZulu-Natal,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Municipality:,NaN,NaN,DC27 - uMkhanyakude,NaN,NaN,NaN,NaN,NaN,NaN,NaN



2021 non-null values by column:
Unnamed: 0      0
Unnamed: 1     22
Unnamed: 2      0
Unnamed: 3      0
Unnamed: 4     20
Unnamed: 5      2
Unnamed: 6     23
Unnamed: 7      4
Unnamed: 8     21
Unnamed: 9      0
Unnamed: 10     4
Unnamed: 11    23
Unnamed: 12    21
Unnamed: 13    23
Unnamed: 14    21
Unnamed: 15     0
Unnamed: 16    23
Unnamed: 17    21
dtype: int64

Initial municipality data inspection: PASSED


In [4]:
#4. — Check Municipality Coverage
# Purpose:
# Check whether the election files contain all 44 KZN municipalities
# before using them in the municipality-context dataset.

population_municipalities = set(
    population_demographics_df["municipality"]
    .dropna()
    .astype(str)
    .str.strip()
)

election_2016_municipalities = set(
    election_2016_df["Municipality"]
    .dropna()
    .astype(str)
    .str.strip()
)

print("Population demographics municipalities:", len(population_municipalities))
print("2016 election municipalities:", len(election_2016_municipalities))
print()

print("2016 municipalities found:")
print(sorted(election_2016_municipalities))
print()

missing_2016 = population_municipalities - election_2016_municipalities

print("Municipalities missing from 2016 election file:", len(missing_2016))
print(sorted(missing_2016))

print()
print("Municipality coverage inspection completed.")

Population demographics municipalities: 44
2016 election municipalities: 1

2016 municipalities found:
['ETH - eThekwini']

Municipalities missing from 2016 election file: 44
['Abaqulusi Local Municipality', 'Alfred Duma Local Municipality', 'Big Five Hlabisa Local Municipality', 'Dannhauser Local Municipality', 'Dr Nkosazana Dlamini Zuma Local Municipality', 'Emadlangeni Local Municipality', 'Endumeni Local Municipality', 'Ethekwini Metropolitan Municipality', 'Greater Kokstad Local Municipality', 'Impendle Local Municipality', 'Inkosi Langalibalele Local Municipality', 'Jozini Local Municipality', 'KwaDukuza Local Municipality', 'Mandeni Local Municipality', 'Maphumulo Local Municipality', 'Mfolozi Local Municipality', 'Mkhambathini Local Municipality', 'Mpofana Local Municipality', 'Msinga Local Municipality', 'Mthonjaneni Local Municipality', 'Mtubatuba Local Municipality', 'Ndwedwe Local Municipality', 'Newcastle Local Municipality', 'Nkandla Local Municipality', 'Nongoma Local Mu

In [5]:
#5. — Validate Municipality Demographics
# Purpose:
# Verify the 44 municipality demographic records before using them
# as the core municipality-level context dataset.

print("Municipality demographics validation:")
print(f"Rows: {len(population_demographics_df):,}")
print(f"Unique municipalities: {population_demographics_df['municipality'].nunique():,}")
print()

print("Duplicate municipality records:")
print(
    population_demographics_df
    .duplicated(subset=["municipality"])
    .sum()
)

print()
print("Missing values by column:")
print(population_demographics_df.isna().sum())

print()
print("Population summary:")
print(population_demographics_df["population"].describe())

assert len(population_demographics_df) == 44
assert population_demographics_df["municipality"].nunique() == 44
assert population_demographics_df.duplicated(
    subset=["municipality"]
).sum() == 0
assert population_demographics_df["population"].notna().all()
assert (population_demographics_df["population"] >= 0).all()

print()
print("Municipality demographics validation: PASSED")

Municipality demographics validation:
Rows: 44
Unique municipalities: 44

Duplicate municipality records:
0

Missing values by column:
municipality                  0
household                     0
homeless                      0
transient                     0
institution                   0
urban_area                    0
tribal_or_traditional_area    0
farm_area                     0
male                          0
population                    0
dtype: int64

Population summary:
count    4.400000e+01
mean     2.823615e+05
std      6.264800e+05
min      3.338236e+04
25%      1.104615e+05
50%      1.539917e+05
75%      2.228024e+05
max      4.239901e+06
Name: population, dtype: float64

Municipality demographics validation: PASSED


In [6]:
#6. — Build Municipality Context Panel
# Purpose:
# Use the validated KZN municipality demographics as the core
# municipality-level context dataset.

municipality_context = population_demographics_df.copy()

municipality_context = municipality_context.rename(
    columns={
        "municipality": "Municipality",
        "household": "Household",
        "homeless": "Homeless",
        "transient": "Transient",
        "institution": "Institution",
        "urban_area": "UrbanArea",
        "tribal_or_traditional_area": "TribalOrTraditionalArea",
        "farm_area": "FarmArea",
        "male": "MalePopulation",
        "population": "Population"
    }
)

print("Municipality context panel created.")
print(f"Rows: {len(municipality_context):,}")
print(f"Columns: {len(municipality_context.columns):,}")
print()

print("Municipalities:", municipality_context["Municipality"].nunique())
print()

display(municipality_context.head())

assert len(municipality_context) == 44
assert municipality_context["Municipality"].nunique() == 44

print()
print("Municipality context panel: PASSED")

Municipality context panel created.
Rows: 44
Columns: 10

Municipalities: 44



,Municipality,Household,Homeless,Transient,Institution,UrbanArea,TribalOrTraditionalArea,FarmArea,MalePopulation,Population
0,Umdoni Local Municipality,99,1,262,39850.271426,104607.218618,11985.390218,74391.521503,82051.358758,156442.880261
1,Umzumbe Local Municipality,56,4,125,0.000000,139029.363876,15.442094,65512.792209,73532.013761,139044.805970
2,UMuziwabantu Local Municipality,54,3,183,12871.211719,100534.222489,2374.091303,54516.210659,61263.314852,115779.525511
3,Ray Nkonyeni Local Municipality,204,1,998,97837.430403,254636.931174,9660.081033,171980.174616,190154.267993,362134.442609
4,uMshwathi Local Municipality,10,1,605,15332.360949,72093.563278,31051.908503,55836.571936,62641.260794,118477.832730



Municipality context panel: PASSED


In [7]:
#7. — Validate Municipality Context Panel
# Purpose:
# Verify the final municipality context panel before saving
# the KZN municipality-level dataset.

print("Municipality context validation:")
print(f"Rows: {len(municipality_context):,}")
print(f"Unique municipalities: {municipality_context['Municipality'].nunique():,}")
print()

print("Duplicate municipality records:")
print(
    municipality_context
    .duplicated(subset=["Municipality"])
    .sum()
)

print()
print("Missing values by column:")
print(municipality_context.isna().sum())

print()
print("Population summary:")
print(municipality_context["Population"].describe())

assert len(municipality_context) == 44
assert municipality_context["Municipality"].nunique() == 44
assert municipality_context.duplicated(
    subset=["Municipality"]
).sum() == 0
assert municipality_context["Population"].notna().all()
assert (municipality_context["Population"] >= 0).all()

print()
print("Municipality context validation: PASSED")

Municipality context validation:
Rows: 44
Unique municipalities: 44

Duplicate municipality records:
0

Missing values by column:
Municipality               0
Household                  0
Homeless                   0
Transient                  0
Institution                0
UrbanArea                  0
TribalOrTraditionalArea    0
FarmArea                   0
MalePopulation             0
Population                 0
dtype: int64

Population summary:
count    4.400000e+01
mean     2.823615e+05
std      6.264800e+05
min      3.338236e+04
25%      1.104615e+05
50%      1.539917e+05
75%      2.228024e+05
max      4.239901e+06
Name: Population, dtype: float64

Municipality context validation: PASSED


In [8]:
#8. — Save Municipality Context Panel
# Purpose:
# Save the validated municipality context panel so it can be
# reused as a processed KZN municipality-level dataset.

municipality_context.to_csv(
    MUNICIPALITY_OUTPUT,
    index=False
)

print("Municipality context panel saved.")
print(f"Output file: {MUNICIPALITY_OUTPUT}")
print(f"Rows saved: {len(municipality_context):,}")
print(f"Columns saved: {len(municipality_context.columns):,}")

assert MUNICIPALITY_OUTPUT.exists()

print()
print("Save validation: PASSED")

Municipality context panel saved.
Output file: C:\Users\Student\Downloads\Big Data\SPU-TEAM-DIRISA\data\processed\03_municipality_context\municipality_context.csv
Rows saved: 44
Columns saved: 10

Save validation: PASSED
